In [1]:
import os
import glob
import time
from datetime import datetime, date
import ee
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import rasterio
from rasterio.mask import mask
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import scipy.stats as stats
from datetime import timedelta
import geopandas as gpd
import folium
import branca.colormap as cm
import io
import base64
from PIL import Image
from shapely.geometry import mapping

# =====================================================================
# STEP 1 & 2: HARDWARE ALIGNMENT & GEE ORBITAL EXTRACTION (1 KM)
# =====================================================================
os.environ.update({
    "OMP_NUM_THREADS": "1", "OPENBLAS_NUM_THREADS": "1", 
    "MKL_NUM_THREADS": "1", "VECLIB_MAXIMUM_THREADS": "1", "NUMEXPR_NUM_THREADS": "1"
})

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" -> Hardware targeting: {device}")

GCP_PROJECT_ID = 'indigo-syntax-420618'
ee.Initialize(project=GCP_PROJECT_ID)

base_dir = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Production"
model_dir = os.path.join(base_dir, "Models")
pred_dir = os.path.join(base_dir, "Prediction")
data_dir = os.path.join(base_dir, "Data")
sliced_dir = os.path.join(base_dir, "Sliced_Data")

os.makedirs(pred_dir, exist_ok=True)
os.makedirs(data_dir, exist_ok=True)

mt_state = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM1_NAME', 'Mato Grosso'))
roi_bounds = mt_state.geometry().bounds()
region_coords = roi_bounds.getInfo()['coordinates']

current_year = datetime.now().year
today = date.today()
dynamic_prefixes = ['ERA5_Precip', 'ERA5_Wind']

print(f"\n--- Enforcing fresh daily data for ongoing year ({current_year}) ---")
for prefix in dynamic_prefixes:
    f_path = os.path.join(data_dir, f'{prefix}_{current_year}.tif')
    if os.path.exists(f_path):
        file_date = date.fromtimestamp(os.path.getmtime(f_path))
        if file_date < today:
            print(f"  [OUTDATED] {os.path.basename(f_path)}. Deleting to force GEE update...")
            os.remove(f_path)
            for pt_file in glob.glob(os.path.join(sliced_dir, f"*_patch_{current_year}_*.pt")):
                os.remove(pt_file)

# ---------------------------------------------------------------------
# FIX APPLIED: Resolução da Matriz rebaixada e estipulada em 1 KM
# ---------------------------------------------------------------------
def queue_export(image, description, file_prefix, scale=1000):
    ee.batch.Export.image.toDrive(
        image=image, description=description, folder='Data', fileNamePrefix=file_prefix,
        region=region_coords, scale=scale, crs='EPSG:4326', maxPixels=1e13
    ).start()

missing_files = 0
for year in range(2018, current_year + 1):
    start_date = ee.Date(f'{year}-01-01')
    end_date   = ee.Date(f'{year+1}-01-01')
    
    needs_precip = not os.path.exists(os.path.join(data_dir, f'ERA5_Precip_{year}.tif'))
    needs_wind = not os.path.exists(os.path.join(data_dir, f'ERA5_Wind_{year}.tif'))
    
    if needs_precip or needs_wind:
        era5_collection = ee.ImageCollection("ECMWF/ERA5_LAND/HOURLY").filterBounds(roi_bounds).filterDate(start_date, end_date)
        days_list = ee.List.sequence(1, start_date.advance(1, 'year').difference(start_date, 'day'))
        
        if needs_precip:
            print(f"  [MISSING] Queueing 1 KM Precipitation for {year}...")
            def aggregate_precip(day_offset):
                t_day = start_date.advance(ee.Number(day_offset).subtract(1), 'day')
                d_col = era5_collection.filterDate(t_day, t_day.advance(1, 'day'))
                d_sum = ee.Image(ee.Algorithms.If(d_col.size().gt(0), d_col.select('total_precipitation').sum().multiply(1000).toFloat(), ee.Image.constant(0).rename('total_precipitation').toFloat()))
                return d_sum.rename(ee.String('day_').cat(ee.Number(day_offset).format('%d'))).set('system:time_start', t_day.millis())
            queue_export(ee.ImageCollection.fromImages(days_list.map(aggregate_precip)).toBands(), f'ERA5_Precip_{year}', f'ERA5_Precip_{year}')
            missing_files += 1
            
        if needs_wind:
            print(f"  [MISSING] Queueing 1 KM Wind Vectors for {year}...")
            def aggregate_wind(day_offset):
                t_day = start_date.advance(ee.Number(day_offset).subtract(1), 'day')
                d_col = era5_collection.filterDate(t_day, t_day.advance(1, 'day'))
                d_mean = ee.Image(ee.Algorithms.If(d_col.size().gt(0), ee.Image.cat([d_col.select('u_component_of_wind_10m').mean().toFloat(), d_col.select('v_component_of_wind_10m').mean().toFloat()]), ee.Image.cat([ee.Image.constant(0), ee.Image.constant(0)]).rename(['u_component_of_wind_10m', 'v_component_of_wind_10m']).toFloat()))
                return d_mean.rename(['u_wind', 'v_wind']).set('system:time_start', t_day.millis())
            queue_export(ee.ImageCollection.fromImages(days_list.map(aggregate_wind)).toBands(), f'ERA5_Wind_{year}', f'ERA5_Wind_{year}')
            missing_files += 1

if missing_files > 0:
    print(f"\n [!] Tasks sent to GEE. Wait for Google Drive sync before running the next cell.")
else:
    print(f"\n [OK] All meteorological files present. Ready for CNN inference.")

 -> Hardware targeting: cuda

--- Enforcing fresh daily data for ongoing year (2026) ---

 [OK] All meteorological files present. Ready for CNN inference.


In [2]:
# =====================================================================
# STEP 3 & 4: CNN ARCHITECTURE & TENSOR IMPUTATION (x_input)
# =====================================================================
class DeepFireCNN(nn.Module):
    def __init__(self, in_ch=75):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1), nn.GroupNorm(8, 64), nn.ReLU(),
            nn.Conv2d(64, 128, 3, padding=1), nn.GroupNorm(8, 128), nn.ReLU(),
            nn.Conv2d(128, 256, 3, padding=1), nn.GroupNorm(8, 256), nn.ReLU(),
            nn.Dropout2d(0.2), 
            nn.Conv2d(256, 1, 1)
        )
    def forward(self, x): return torch.sigmoid(self.net(x))

model_path = os.path.join(model_dir, "final_model_1d_period.pth")
model = DeepFireCNN(in_ch=75).to(device)
model.load_state_dict(torch.load(model_path, map_location=device))
model.eval()

opt_threshold = 0.75 
print(f" -> CNN Loaded. Stringency threshold explicitly set to: {opt_threshold}")

target_date_str = input("\n[INPUT] Enter the target date to predict (DD/MM/YYYY): ")
target_date = datetime.strptime(target_date_str, "%d/%m/%Y")
target_year = target_date.year
target_day_of_year = target_date.timetuple().tm_yday

f_wind = os.path.join(data_dir, f'ERA5_Wind_{target_year}.tif')
f_precip = os.path.join(data_dir, f'ERA5_Precip_{target_year}.tif')

# T-TEST IMPUTATION FOR x_input GENERATION (The 14-day history)
def get_latest_patch_for_year(year):
    files = glob.glob(os.path.join(sliced_dir, f"*patch_{year}_*.pt"))
    return torch.load(max(files, key=os.path.getmtime), weights_only=False) if files else None

current_year_data = get_latest_patch_for_year(target_year)
historical_years = [y for y in range(2018, target_year)]
hist_cache = {hy: get_latest_patch_for_year(hy) for hy in historical_years if get_latest_patch_for_year(hy)}

available_days = min(14, current_year_data['time_steps'] - (target_day_of_year - 14)) if current_year_data else 0
if available_days < 0: available_days = 0

final_env_chunks = []
if available_days == 14:
    t_idx = target_day_of_year - 14
    final_env_chunks = [current_year_data['env'][v, t_idx : t_idx + 14] for v in range(5)]
    lulc_chunk = current_year_data['lulc'][-1].unsqueeze(0)
    base_static = current_year_data
else:
    t_start, t_end = target_day_of_year - 14, (target_day_of_year - 14) + available_days
    for var_idx in range(5):
        if available_days > 0 and current_year_data is not None:
            curr_dist = current_year_data['env'][var_idx, t_start:t_end].mean(dim=(1, 2)).numpy()
        else:
            curr_dist = current_year_data['env'][var_idx, :current_year_data['time_steps']].mean(dim=(1, 2)).numpy() if current_year_data and current_year_data['time_steps'] > 0 else np.array([0,0])

        best_year, best_pval = None, -1.0
        for hy, hdata in hist_cache.items():
            h_dist = hdata['env'][var_idx, t_start:t_end].mean(dim=(1, 2)).numpy()
            if len(curr_dist) >= 2 and len(h_dist) >= 2:
                _, p_val = stats.ttest_ind(curr_dist, h_dist, equal_var=False)
                if p_val > best_pval: best_pval, best_year = p_val, hy
        
        src_year = best_year if best_year else max(hist_cache.keys())
        if available_days > 0 and current_year_data is not None:
            merged = torch.cat([current_year_data['env'][var_idx, t_start : t_start + available_days], hist_cache[src_year]['env'][var_idx, t_start + available_days : t_start + 14]], dim=0)
        else:
            merged = hist_cache[src_year]['env'][var_idx, target_day_of_year - 14 : target_day_of_year]
        final_env_chunks.append(merged)
        
    recent_hy = max(hist_cache.keys())
    lulc_chunk = current_year_data['lulc'][-1].unsqueeze(0) if current_year_data and current_year_data['time_steps'] > 0 else hist_cache[recent_hy]['lulc'][-1].unsqueeze(0)
    base_static = current_year_data if current_year_data is not None else hist_cache[recent_hy]

x_input = torch.cat(final_env_chunks + [lulc_chunk, base_static['roads'].unsqueeze(0), base_static['dem'].unsqueeze(0), base_static['slope'].unsqueeze(0), base_static['aspect'].unsqueeze(0)], dim=0).unsqueeze(0).to(device)
print(f"[OK] Temporal Tensor generated cleanly: {x_input.shape}")

 -> CNN Loaded. Stringency threshold explicitly set to: 0.75
[OK] Temporal Tensor generated cleanly: torch.Size([1, 75, 128, 128])


In [3]:
# =====================================================================
# STEP 6 & 7: AUTOREGRESSIVE GPU ENGINE & ADVANCED CARTOGRAPHY
# =====================================================================
print(f"\n{'#'*80}\n STEP 6: AUTOREGRESSIVE CONVERGENCE (CNN + GPU CA)\n{'#'*80}")

SIMULATION_DAYS = 7
BASE_SPREAD_RATE = 0.05
WIND_WEIGHT = 0.4
RAIN_SENSITIVITY = 100.0 

with rasterio.open(f_wind) as src_w:
    meta = src_w.meta.copy()
    u_wind = np.nan_to_num(src_w.read(1)).astype(np.float32)
    v_wind = np.nan_to_num(src_w.read(2)).astype(np.float32)

min_y, min_x = u_wind.shape
meta.update(dtype=rasterio.float32, count=1, nodata=np.nan)

u_wind_t = torch.tensor(u_wind, device=device)
v_wind_t = torch.tensor(v_wind, device=device)
wind_speed = torch.sqrt(u_wind_t**2 + v_wind_t**2)
wind_dir = torch.atan2(v_wind_t, u_wind_t)
del u_wind, v_wind; gc.collect()

neighbors = [(-1, 0, np.pi/2), (-1, 1, np.pi/4), (0, 1, 0.0), (1, 1, -np.pi/4), (1, 0, -np.pi/2), (1, -1, -3*np.pi/4), (0, -1, np.pi), (-1, -1, 3*np.pi/4)]

print(" -> Generating Absolute Day 1 Ground Truth via CNN...")
with torch.no_grad():
    raw_logits_day1 = model(x_input)
day1_probs = torch.sigmoid(raw_logits_day1).squeeze()

# Alinhamento perfeitamente congruente com a matriz 1 KM
day1_probs_aligned = F.interpolate(day1_probs.unsqueeze(0).unsqueeze(0), size=(min_y, min_x), mode='bilinear', align_corners=False).squeeze()
current_state = (day1_probs_aligned >= opt_threshold).float()

for day in range(SIMULATION_DAYS):
    d_eval = target_date + timedelta(days=day)
    d_yday = d_eval.timetuple().tm_yday
    
    with torch.no_grad():
        injected_probs = torch.sigmoid(model(x_input)).squeeze()
    injected_aligned = F.interpolate(injected_probs.unsqueeze(0).unsqueeze(0), size=(min_y, min_x), mode='bilinear').squeeze()
    cnn_injected_ignitions = (injected_aligned >= opt_threshold).float()
    
    with rasterio.open(f_precip) as src_p:
        rain_matrix = np.nan_to_num(src_p.read(d_yday)).astype(np.float32) if d_yday <= src_p.count else np.zeros((min_y, min_x), dtype=np.float32)
            
    rain_dampening = torch.clamp(1.0 - (torch.tensor(rain_matrix[:min_y, :min_x], device=device) * RAIN_SENSITIVITY), min=0.0, max=1.0)
    next_state = current_state.clone()
    
    for dy, dx, angle in neighbors:
        shifted_state = torch.roll(current_state, shifts=(dy, dx), dims=(0, 1))
        if dy > 0: shifted_state[:dy, :] = 0
        elif dy < 0: shifted_state[dy:, :] = 0
        if dx > 0: shifted_state[:, :dx] = 0
        elif dx < 0: shifted_state[:, dx:] = 0
        
        spread_matrix = BASE_SPREAD_RATE * (1 + WIND_WEIGHT * wind_speed * torch.cos(wind_dir - angle))
        next_state += shifted_state * spread_matrix * rain_dampening
        
    current_state = torch.clamp(next_state + cnn_injected_ignitions, min=0.0, max=1.0)
    
    with rasterio.open(os.path.join(pred_dir, f"Spread_Sim_Day{day+1}_{target_date.strftime('%Y_%m_%d')}.tif"), 'w', **meta) as dst:
        dst.write(current_state.cpu().numpy(), 1)

torch.cuda.empty_cache(); gc.collect()
print("[OK] Physical Simulation Complete.")

# ---------------------------------------------------------------------
# CARTOGRAPHY AND MASKING
# ---------------------------------------------------------------------
print("\n -> Initiating High-Res Vector Masking & Folium Mapping...")
path_mun = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Queimadas_Data/MT_Municipios_2022/MT_Municipios_2022.shp"
path_fed = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Queimadas_Data/Federal Highways/SNV_202410A.shp"

mt_gdf = gpd.read_file(path_mun).to_crs(epsg=4326)
mt_state_boundary = mt_gdf.dissolve()
mt_geom = [mapping(mt_state_boundary.geometry.iloc[0])] 
city_col = 'NM_MUN' if 'NM_MUN' in mt_gdf.columns else mt_gdf.columns[0]
fed_gdf = gpd.clip(gpd.read_file(path_fed).to_crs(epsg=4326), mt_state_boundary)

pred_maps, map_bounds = {}, None
for day in range(SIMULATION_DAYS):
    h = day + 1
    output_tif = os.path.join(pred_dir, f"Spread_Sim_Day{h}_{target_date.strftime('%Y_%m_%d')}.tif")
    with rasterio.open(output_tif) as src_clip:
        if map_bounds is None: map_bounds = [[src_clip.bounds.bottom, src_clip.bounds.left], [src_clip.bounds.top, src_clip.bounds.right]]
        out_image, out_transform = mask(src_clip, mt_geom, crop=True, filled=False)
        out_meta = src_clip.meta.copy()
        out_meta.update({"driver": "GTiff", "height": out_image.shape[1], "width": out_image.shape[2], "transform": out_transform})
    with rasterio.open(output_tif, "w", **out_meta) as dest: dest.write(out_image)
    pred_maps[h] = out_image[0]

# Generating JPG
titles_pt = {d+1: f"Dia +{d+1} ({(target_date + timedelta(days=d)).strftime('%d/%m')})" for d in range(SIMULATION_DAYS)}
cmap_base = mcolors.LinearSegmentedColormap.from_list("BlueYellowRed", ["blue", "yellow", "red"])
cmap_jpg = cmap_base.copy(); cmap_jpg.set_bad(color='white') 

fig, axes = plt.subplots(1, SIMULATION_DAYS, figsize=(6 * SIMULATION_DAYS, 6))
for idx in range(SIMULATION_DAYS):
    im = axes[idx].imshow(pred_maps[idx+1], cmap=cmap_jpg, vmin=0, vmax=1)
    axes[idx].set_title(titles_pt[idx+1], fontsize=14, fontweight='bold', pad=10); axes[idx].axis('off')
cbar_ax = fig.add_axes([0.15, 0.05, 0.7, 0.03])
fig.colorbar(im, cax=cbar_ax, orientation='horizontal', label="Probabilidade Acumulada de Fogo")
plt.suptitle(f"Evolução Diária - Autoregressivo - {target_date.strftime('%d/%m/%Y')}", fontsize=18, fontweight='bold', y=0.98)
plt.savefig(os.path.join(pred_dir, f"Painel_Autoregressivo_{target_date.strftime('%Y_%m_%d')}.jpg"), dpi=600, bbox_inches='tight', facecolor='white')
plt.close(fig)

# Generating Interactive HTML
cmap_html = cmap_base.copy(); cmap_html.set_bad(alpha=0.0) 
interactive_map = folium.Map(location=[-12.64, -55.42], zoom_start=6, tiles="CartoDB positron", control_scale=True)
interactive_map.add_child(cm.LinearColormap(colors=['blue', 'yellow', 'red'], vmin=0.0, vmax=1.0).add_to(interactive_map))

for h in range(1, SIMULATION_DAYS + 1):
    colorized = cmap_html(np.ma.masked_invalid(pred_maps[h]))
    buffered = io.BytesIO()
    Image.fromarray(np.uint8(colorized * 255)).save(buffered, format="PNG")
    folium.raster_layers.ImageOverlay(image=f"data:image/png;base64,{base64.b64encode(buffered.getvalue()).decode()}", bounds=map_bounds, opacity=0.75, name=titles_pt[h], show=(h == 1)).add_to(interactive_map)

folium.GeoJson(mt_gdf, name="Limites", style_function=lambda x: {'color': '#333333', 'weight': 0.5, 'fillOpacity': 0}).add_to(interactive_map)
folium.GeoJson(fed_gdf, name="Rodovias", style_function=lambda x: {'color': 'green', 'weight': 1}).add_to(interactive_map)
folium.LayerControl(collapsed=False).add_to(interactive_map)
interactive_map.save(os.path.join(pred_dir, f"Mapa_Autoregressivo_{target_date.strftime('%Y_%m_%d')}.html"))

print(f"[SUCCESS] Pipeline Finalizado. Arquivos salvos em: {pred_dir}")


################################################################################
 STEP 6: AUTOREGRESSIVE CONVERGENCE (CNN + GPU CA)
################################################################################
 -> Generating Absolute Day 1 Ground Truth via CNN...
[OK] Physical Simulation Complete.

 -> Initiating High-Res Vector Masking & Folium Mapping...
[SUCCESS] Pipeline Finalizado. Arquivos salvos em: /home/daves/google_drive/Pessoal/Notebooks/Queimadas/Production/Prediction
